# Spirula Cloud — yüksek kaliteli video / fotoğraf rekonstrüksiyonu

Bu notebook ham bir videoyu (veya hazırlanmış bir COLMAP veri kümesini) Drive’dan alır; kareleri seçer, Spirula ile SfM yapar, en büyük geometri modelini çalıştırır ve son olarak **ultra** splat eğitimi başlatır. Varsayılan video yolu örnektir; ilk ayar hücresinde gerçek dosya yolunu düzenle. `video`, `dataset_zip` ve `dataset_folder` giriş kipleri vardır.

Sabit ikili resmî `v2026.9.24` Linux Vulkan sürümüdür. Önceden derlenmiş ikili kullanılır; notebook Spirula’yı kaynak koddan derlemez ve NVIDIA kernel sürücüsünü kurmaz/güncellemez. 580.82.07 sürücüsünde NVIDIA Vulkan kullanıcı alanı eksikse, doğrulanan NVIDIA runfile yalnızca çalışma klasörüne açılıp grafik kütüphaneleri işlem ortamına eklenir. Başka sürücü sürümleri için eşleşmeyen dosya kullanılmaz.

Maksimum iş yükü ayarları: video için 4 fps hedefi ve hareket uyarlamalı örnekleme, SfM `extreme`, MoGe2 ViT-L (en çok 3600 token; modelin kendi pipeline değeri 1064 piksel), eğitim `ultra` (80.000 adım ve 10 milyon splat). Tam ultra eğitim için en az 38 GiB görünür GPU belleği aranır; daha düşük bellekte kalite bütçesi kendiliğinden düşürülmez. En büyük modelden yalnızca 80%’den az görüntü kayıtlıysa veya büyük bağlı bileşen verinin 80%’ini kapsamıyorsa notebook uzun eğitimi durdurur; ayar hücresindeki açık geçersiz kılma bunu değiştirebilir.

Uzun eğitim Colab oturumunun süresini aşabilir. Her doğrulanmış checkpoint yerel olarak tutulur ve Drive’a en son tam checkpoint olarak eşitlenmeye çalışılır. Drive eşitlemesi başarısız olsa bile yerel günlükler korunur. Nihai PLY ancak eğitim tamamlandıktan ve dosya doğrulandıktan sonra tamamlandı olarak işaretlenir. Colab GPU’sunda bu tam ultra iş yükü henüz çalıştırılmadı.


## 0. Düzenlenebilir kalite ve girdi ayarları


In [ ]:
# Giriş: video (varsayılan), dataset_zip veya dataset_folder.
INPUT_MODE = 'video'
INPUT_VIDEO = '/content/drive/MyDrive/GaussianTests/inputs/IMG_5961.MOV'  # Düzenle
INPUT_DATASET_ZIP = '/content/drive/MyDrive/GaussianTests/inputs/spriula_power_spike.zip'
INPUT_DATASET_FOLDER = '/content/drive/MyDrive/GaussianTests/inputs/prepared_dataset'
CAPTURE_KIND = 'video'  # video veya individual; dataset kipinde SfM gerektiğinde kullanılır

# Kare örnekleme: ffprobe gerçek fps'i ölçer, skip değerini bu hedefe göre hesaplar.
TARGET_FPS = 4.0
FRAME_SKIP_OVERRIDE = None  # İsteğe bağlı: ffprobe yoksa kaynak video kare aralığını elle ver
MAX_EXTRACTED_FRAMES = 0     # 0 = sınır yok; kalite için sessizce kare düşürülmez

# Geometri: MoGe2 ViT-L, MoGe ailesindeki en büyük seçenek. Metric3D Giant2 ayrı bir
# ailedeki en büyük seçenektir; modeller arasında evrensel bir kalite sıralaması yok.
GEOMETRY_MODEL = 'moge2-vitl'  # moge2-vits | moge2-vitb | moge2-vitl | metric3d-vit-small | metric3d-vit-large | metric3d-vit-giant2
GEOMETRY_MAX_SIZE = 1064        # MoGe2'nin kendi pipeline değeridir
GEOMETRY_NUM_TOKENS = 3600      # MoGe2 için belgelenen üst sınır; Metric3D bunu yoksayar
GENERATE_DEPTH = True

# Ultra kalite macro'sunun varsayılanları kaynakta 80.000 adım / 10.000.000 splat.
TRAIN_ITERATIONS = 80000
TRAIN_CAP_MAX = 10000000
MIN_TRAIN_VRAM_GIB = 38.0      # 40-GB GPU'lar nvidia-smi'de yaklaşık 38 GiB görünür
CHECKPOINT_INTERVAL = 10000
MIN_REGISTERED_COVERAGE = 0.80
MIN_DOMINANT_COMPONENT_SHARE = 0.80
ALLOW_PARTIAL_RECONSTRUCTION = False  # Sorunlu/fragmentli SfM sonrası uzun eğitimi açıkça geçersiz kıl
RESUME_CHECKPOINT = ''          # İsteğe bağlı: Drive'daki tam step-XXXXXXXXX.ckpt klasörünün yolu


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib
import json
import os
import platform
import re
import shutil
import stat
import struct
import subprocess
import time
import uuid
import zipfile
from datetime import datetime
from pathlib import Path, PurePosixPath

DATASET_ZIP = Path(INPUT_DATASET_ZIP)
VIDEO_PATH = Path(INPUT_VIDEO)
DATASET_FOLDER = Path(INPUT_DATASET_FOLDER)
RELEASE_TAG = 'v2026.9.24'
VERSION = '2026.9.24'
COMMIT = '183b2c6df72f42ecb9a0500cdad96749847da171'
ASSET_NAME = 'spirula-2026.9.24-ubuntu-vulkan-x86_64.zip'
ASSET_URL = 'https://github.com/harry7557558/spirula-studio/releases/download/v2026.9.24/spirula-2026.9.24-ubuntu-vulkan-x86_64.zip'
ASSET_SIZE = 51076566
EXPECTED_SHA256 = '3560119e1d6ff91fa4bf49ed6a196f8b092db700e252f6288aa57a7568a65143'
RELEASE_API = 'https://api.github.com/repos/harry7557558/spirula-studio/releases/tags/v2026.9.24'

RUN_ID = 'spirula_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
WORK = Path('/content/spirula_cloud_tests') / RUN_ID
DRIVE_OUT = Path('/content/drive/MyDrive/GaussianTests/spirula_cloud_tests') / RUN_ID
LOGS = WORK / 'logs'
MANIFEST_PATH = WORK / 'manifest.json'
WORK.mkdir(parents=True, exist_ok=False)
LOGS.mkdir(parents=True, exist_ok=False)
OUT = WORK / 'outputs'
BINARY = WORK / 'spirula'

manifest = {
    'run_id': RUN_ID,
    'release_tag': RELEASE_TAG,
    'source_commit': COMMIT,
    'asset_name': ASSET_NAME,
    'expected_sha256': EXPECTED_SHA256,
    'input_mode': INPUT_MODE,
    'dataset_zip': str(DATASET_ZIP),
    'input_video': str(VIDEO_PATH),
    'input_dataset_folder': str(DATASET_FOLDER),
    'status': 'starting',
    'drive_export_status': 'pending',
    'local_work_dir': str(WORK),
    'local_logs_dir': str(LOGS),
    'local_manifest': str(MANIFEST_PATH),
    'local_outputs_dir': str(OUT),
    'drive_export_dir': str(DRIVE_OUT),
    'created_at_utc': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
}

def save_manifest():
    temp = MANIFEST_PATH.with_suffix('.tmp')
    temp.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    temp.replace(MANIFEST_PATH)


def _copy_file_to_drive(source, destination, label):
    try:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        return True, None
    except Exception as exc:
        message = f'{label}: {type(exc).__name__}: {exc}'
        print(f'Drive eşitlemesi başarısız; yerel dosya korunuyor: {source} ({message})', flush=True)
        return False, message


def _copy_tree_to_drive(source, destination, label):
    if not source.exists():
        return True, None
    try:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, destination, dirs_exist_ok=True)
        return True, None
    except Exception as exc:
        message = f'{label}: {type(exc).__name__}: {exc}'
        print(f'Drive eşitlemesi başarısız; yerel klasör korunuyor: {source} ({message})', flush=True)
        return False, message


def retry_drive_export():
    """Resync local logs, outputs and manifest without letting Drive errors escape."""
    errors = []
    snapshot_sync = globals().get('sync_dataset_snapshot')
    if snapshot_sync:
        errors.extend(snapshot_sync())
    ok, error = _copy_tree_to_drive(LOGS, DRIVE_OUT / 'logs', 'logs')
    if not ok:
        errors.append(error)
    ok, error = _copy_tree_to_drive(OUT, DRIVE_OUT / 'outputs', 'outputs')
    if not ok:
        errors.append(error)

    training_done = manifest.get('training_status') == 'complete'
    if training_done and not errors:
        missing = [entry['path'] for entry in manifest.get('ply_files', [])
                   if not (DRIVE_OUT / 'outputs' / entry['path']).is_file()]
        if missing:
            errors.append('Drive outputs are missing PLY files: ' + ', '.join(missing))
            print('Drive kopyası eksik PLY içeriyor; yerel çıktılar korunuyor:', ', '.join(missing), flush=True)
    manifest['drive_export_status'] = 'pending' if errors else 'complete'
    manifest['drive_export_errors'] = errors
    if training_done:
        manifest['status'] = 'export_pending' if errors else 'complete'
    save_manifest()

    ok, error = _copy_file_to_drive(MANIFEST_PATH, DRIVE_OUT / 'manifest.json', 'manifest')
    if not ok:
        errors.append(error)
        manifest['drive_export_status'] = 'pending'
        manifest['drive_export_errors'] = errors
        if training_done:
            manifest['status'] = 'export_pending'
        save_manifest()

    if errors:
        print('Drive kopyası beklemede. Yerel sonuçlar korunuyor; Drive bağlandıktan sonra son “Drive’a aktarımı yeniden dene” hücresini çalıştır.', flush=True)
        print('Yerel günlükler:', LOGS)
        print('Yerel manifest:', MANIFEST_PATH)
        if OUT.exists():
            print('Yerel çıktılar:', OUT)
    elif training_done:
        print('Sonuçlar Drive’a aktarıldı:', DRIVE_OUT)
    return errors


def fail(message, log_name='preflight'):
    log_path = LOGS / (log_name + '.log')
    with log_path.open('a', encoding='utf-8') as f:
        f.write(str(message) + '\n')
    manifest['status'] = 'failed'
    manifest['error'] = str(message)
    save_manifest()
    retry_drive_export()
    raise RuntimeError(f'{message}\nGünlük: {log_path}')


def require(condition, message, log_name='preflight'):
    if not condition:
        fail(message, log_name)


def run_logged(name, args, cwd=None, timeout=600, accepted_exit_codes=(0,)):
    log_path = LOGS / (name + '.log')
    rendered = [str(a) for a in args]
    started = time.monotonic()
    proc = None
    problem = None
    with log_path.open('w', encoding='utf-8') as log:
        log.write('COMMAND: ' + repr(rendered) + '\n')
        log.flush()
        try:
            proc = subprocess.Popen(rendered, cwd=cwd, stdout=log, stderr=subprocess.STDOUT, text=True)
            last_note = started - 30
            while proc.poll() is None:
                now = time.monotonic()
                if now - started > timeout:
                    raise TimeoutError(f'{name}: {timeout} saniyelik süre sınırı aşıldı.')
                if now - last_note >= 30:
                    print(f'{name}: {now-started:.0f} saniye; günlük yerel diske yazılıyor.', flush=True)
                    last_note = now
                time.sleep(2)
            if proc.returncode not in accepted_exit_codes:
                raise RuntimeError(f'{name}: process exited with code {proc.returncode}')
        except BaseException as exc:
            problem = exc
            if proc is not None and proc.poll() is None:
                proc.terminate()
                try:
                    proc.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait()
            log.write('ERROR: ' + repr(exc) + '\n')
            log.flush()
    log_text = log_path.read_text(encoding='utf-8', errors='replace')
    if problem is not None:
        tail = '\n'.join(log_text.splitlines()[-40:])
        manifest['status'] = 'failed'
        manifest['error'] = f'{name}: {problem}'
        manifest['failed_step'] = name
        manifest['local_failed_log'] = str(log_path)
        save_manifest()
        print(f'{name} failed: {problem}. Last 40 log lines:\n{tail}', flush=True)
        retry_drive_export()
        raise RuntimeError(f'{name} failed: {problem}\nLocal log: {log_path}\nLast 40 log lines:\n{tail}') from problem
    return log_text

save_manifest()
print('Test:', RUN_ID)
print('Yerel çalışma klasörü:', WORK)
print('Drive sonuç klasörü (eşitleme başarılıysa):', DRIVE_OUT)
require(INPUT_MODE in ('video', 'dataset_zip', 'dataset_folder'),
        f'INPUT_MODE geçersiz: {INPUT_MODE}. video, dataset_zip veya dataset_folder seç.', 'input')
try:
    if INPUT_MODE == 'video':
        input_exists = VIDEO_PATH.is_file()
        input_description = VIDEO_PATH
    elif INPUT_MODE == 'dataset_zip':
        input_exists = DATASET_ZIP.is_file()
        input_description = DATASET_ZIP
    else:
        input_exists = DATASET_FOLDER.is_dir()
        input_description = DATASET_FOLDER
except Exception as exc:
    fail(f'Giriş dosyasına Drive üzerinden erişilemiyor: {type(exc).__name__}: {exc}', 'input')
require(input_exists, f'Giriş bulunamadı: {input_description}. İlk ayar hücresindeki yolu kontrol et.')


## 1. Linux ikilisi ve GPU/Vulkan çalışma zamanı


In [ ]:
# Önce GLIBC sürümünü kontrol et. Eski libc'yi değiştirmek Colab'i bozabilir;
# bu notebook libc veya NVIDIA sürücüsü kurmaz ya da yükseltmez.
libc_name, libc_version = platform.libc_ver()
match = re.match(r'^(\d+)\.(\d+)', libc_version)
libc_tuple = tuple(map(int, match.groups())) if match else (0, 0)
require(libc_name == 'glibc' and libc_tuple >= (2, 38),
        f'Bu Spirula ikilisi GLIBC 2.38 veya yenisini ister; runtime {libc_name} {libc_version} bildiriyor. Daha yeni Ubuntu runtime seç; sistem GLIBC paketini değiştirme.',
        'runtime')
manifest['glibc'] = {'name': libc_name, 'version': libc_version}

nvidia_smi = shutil.which('nvidia-smi')
require(nvidia_smi is not None, 'NVIDIA GPU görünmüyor. Colab çalışma zamanı menüsünden GPU seçip oturumu yeniden başlat.', 'gpu')
gpu_info = run_logged('gpu', [nvidia_smi, '--query-gpu=uuid,name,driver_version,memory.total', '--format=csv,noheader'], timeout=30)
require(bool(gpu_info.strip()), 'nvidia-smi GPU bilgisi döndürmedi. GPU günlüğünü incele.', 'gpu')
manifest['nvidia_smi'] = gpu_info.strip().splitlines()[-1]
print('Sistem GLIBC:', libc_version)
print('NVIDIA aygıtı:', manifest['nvidia_smi'])


## 2. Resmî önceden derlenmiş Spirula ikilisi


In [ ]:
# Yalnızca çalışma zamanı paylaşımlı kütüphaneleri eklenir. --no-upgrade,
# sistem paketlerini yükseltmez; CUDA/NVIDIA sürücü paketleri istenmez.
run_logged('apt_update', ['apt-get', 'update', '-qq'], timeout=600)
run_logged('runtime_packages', ['apt-get', 'install', '-y', '--no-upgrade', '--no-install-recommends',
                                'libgomp1', 'libvulkan1', 'libopengl0', 'libstdc++6', 'vulkan-tools'], timeout=900)

import urllib.request


def select_release_asset(release):
    if release.get('tag_name') != RELEASE_TAG:
        raise ValueError(f'Beklenen release etiketi {RELEASE_TAG}, gelen {release.get("tag_name")!r}.')
    assets = [item for item in release.get('assets', []) if item.get('name') == ASSET_NAME]
    if len(assets) != 1:
        raise ValueError(f'{ASSET_NAME} için tam bir release asset bekleniyordu; bulunan: {len(assets)}.')
    asset = assets[0]
    if asset.get('browser_download_url') != ASSET_URL:
        raise ValueError('Release asset indirme URL değeri sabitlenen resmî URL ile eşleşmiyor.')
    if asset.get('size') != ASSET_SIZE:
        raise ValueError(f'Release asset boyutu değişmiş: {asset.get("size")} (beklenen {ASSET_SIZE}).')
    api_digest = asset.get('digest')
    if api_digest and api_digest.lower() != 'sha256:' + EXPECTED_SHA256:
        raise ValueError(f'GitHub API SHA-256 sabit değerle eşleşmiyor: {api_digest}.')
    return asset

try:
    request = urllib.request.Request(RELEASE_API, headers={
        'Accept': 'application/vnd.github+json',
        'User-Agent': 'Spirula-Colab-smoke-notebook',
    })
    with urllib.request.urlopen(request, timeout=30) as response:
        release = json.loads(response.read().decode('utf-8'))
    asset = select_release_asset(release)
except Exception as exc:
    fail(f'GitHub Release metaverisi doğrulanamadı: {exc}', 'release_metadata')

api_digest = asset.get('digest')
manifest['release_asset'] = {
    'name': asset['name'],
    'url': asset['browser_download_url'],
    'size': asset['size'],
    'api_digest': api_digest,
}

archive = WORK / ASSET_NAME
partial = archive.with_suffix('.download')
hasher = hashlib.sha256()
try:
    with urllib.request.urlopen(asset['browser_download_url'], timeout=60) as response, partial.open('wb') as output:
        while True:
            block = response.read(1024 * 1024)
            if not block:
                break
            output.write(block)
            hasher.update(block)
    downloaded_size = partial.stat().st_size
    actual_sha256 = hasher.hexdigest()
    require(downloaded_size == ASSET_SIZE, f'Arşiv boyutu beklenmiyor: {downloaded_size} bayt.', 'release_download')
    require(actual_sha256 == EXPECTED_SHA256, f'SHA-256 doğrulaması başarısız: {actual_sha256}.', 'release_download')
    partial.replace(archive)
except Exception as exc:
    partial.unlink(missing_ok=True)
    if manifest.get('status') != 'failed':
        fail(f'Resmi arşiv indirilemedi veya SHA-256 doğrulanamadı: {exc}', 'release_download')
    raise

manifest['downloaded_sha256'] = actual_sha256
manifest['binary_version'] = VERSION

with zipfile.ZipFile(archive) as zf:
    infos = zf.infolist()
    require(len(infos) == 1 and not infos[0].is_dir(), 'Resmî arşivde tam bir ikili dosya bekleniyordu.', 'release_archive')
    info = infos[0]
    member = PurePosixPath(info.filename.replace('\\', '/'))
    mode = info.external_attr >> 16
    require(not member.is_absolute() and '..' not in member.parts and member.name == 'spirula',
            f'Arşivde beklenmeyen dosya yolu: {info.filename}', 'release_archive')
    require(not stat.S_ISLNK(mode), 'Resmî arşiv sembolik bağlantı içeriyor.', 'release_archive')
    with zf.open(info) as source, BINARY.open('wb') as dest:
        shutil.copyfileobj(source, dest, length=1024 * 1024)
os.chmod(BINARY, 0o755)

ldd_output = run_logged('binary_dependencies', ['ldd', str(BINARY)], cwd=WORK, timeout=30)
require('not found' not in ldd_output.lower(), 'İkili dosyanın bir paylaşımlı kütüphanesi bulunamadı; binary_dependencies.log dosyasına bak.', 'binary_dependencies')
version_output = run_logged('version', [str(BINARY), '--version'], cwd=WORK, timeout=30)
require(VERSION in version_output and COMMIT[:7] in version_output,
        f'İkili sürümü beklenen {VERSION} / {COMMIT[:7]} ile eşleşmiyor.', 'version')
help_output = run_logged('train_help', [str(BINARY), 'train', '--help-all'], cwd=WORK, timeout=30)
required_flags = [
    '--device', '--data', '--colmap-recon-dir', '--output-dir-prefix', '--output-dir-name',
    '--num-iterations', '--steps-per-save', '--cap-max', '--train-resolution-divisor',
    '--cache-images', '--disable-viewer', '--keep-viewer-alive', '--load-normals',
    '--load-depths', '--normal-supervision-weight',
]
missing_flags = [flag for flag in required_flags if flag not in help_output]
require(not missing_flags, 'Sabit sürüm yardımında beklenen eğitim seçenekleri yok: ' + ', '.join(missing_flags), 'train_help')
print('İkili sürüm ve eğitim seçenekleri doğrulandı.')


## 3. NVIDIA Vulkan aygıtını doğrula ve gerekiyorsa tam sürüm kullanıcı alanını aç


In [ ]:
# Standalone Colab cell: diagnose and, only when possible, expose an installed NVIDIA Vulkan ICD.
import glob
import hashlib
import json
import os
import re
import shutil
import subprocess
from pathlib import Path

RECOVERY_WORK = Path(globals().get('WORK', '/content/spirula_cloud_tests/vulkan_recovery'))
RECOVERY_WORK.mkdir(parents=True, exist_ok=True)
RECOVERY_LOG = RECOVERY_WORK / 'vulkan_recovery.log'
NVIDIA_DRIVER_VERSION = '580.82.07'
NVIDIA_RUNFILE_URL = 'https://download.nvidia.com/XFree86/Linux-x86_64/580.82.07/NVIDIA-Linux-x86_64-580.82.07-no-compat32.run'
NVIDIA_RUNFILE_SHA256 = 'c8a3c1306a6abb1d9856d966b42dc8ebd0d5f76275a34e59650d9c14e57cc0c4'
NVIDIA_RUNFILE_SIZE = 325016549

def run_diagnostic(args, show=True, timeout=120, cwd=None, env=None):
    result = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            check=False, timeout=timeout, cwd=cwd, env=env)
    output = result.stdout or ''
    with RECOVERY_LOG.open('a', encoding='utf-8') as log:
        log.write('$ ' + ' '.join(map(str, args)) + '\n' + output + '\n')
    if show:
        print(output.rstrip())
    return result.returncode, output

print('Local diagnostic log:', RECOVERY_LOG)
smi = shutil.which('nvidia-smi')
if not smi:
    raise RuntimeError('nvidia-smi is missing. Select a Colab NVIDIA GPU runtime; no GPU driver was installed or changed.')
smi_code, smi_output = run_diagnostic([smi, '--query-gpu=uuid,name,driver_version,memory.total', '--format=csv,noheader'])
if smi_code:
    raise RuntimeError(f'nvidia-smi failed with exit code {smi_code}; see {RECOVERY_LOG}')

ldconfig = shutil.which('ldconfig')
ldconfig_output = ''
if ldconfig:
    _, ldconfig_output = run_diagnostic([ldconfig, '-p'], show=False)
sonames = ('libEGL_nvidia.so.0', 'libGLX_nvidia.so.0', 'libvulkan_nvidia.so.0')
library_paths = {name: [] for name in sonames}
for line in ldconfig_output.splitlines():
    for name in sonames:
        if name in line and '=>' in line:
            candidate = Path(line.rsplit('=>', 1)[1].strip())
            if candidate.is_file() and candidate not in library_paths[name]:
                library_paths[name].append(candidate.resolve())
for name in sonames:
    if not library_paths[name]:
        for pattern in (f'/usr/lib/**/{name}', f'/usr/local/lib/**/{name}', f'/lib/**/{name}'):
            for match in glob.glob(pattern, recursive=True):
                candidate = Path(match)
                if candidate.is_file() and candidate not in library_paths[name]:
                    library_paths[name].append(candidate.resolve())
print('NVIDIA Vulkan-related libraries:', {k: [str(p) for p in v] for k, v in library_paths.items()})

for variable in ('VK_DRIVER_FILES', 'VK_ICD_FILENAMES', 'VK_ADD_DRIVER_FILES'):
    print(f'{variable}={os.environ.get(variable, "<unset>")}')

home = Path.home()
config_dirs = [Path(os.environ.get('XDG_CONFIG_HOME', home / '.config'))]
config_dirs += [Path(p) for p in os.environ.get('XDG_CONFIG_DIRS', '/etc/xdg').split(':') if p]
data_dirs = [Path(os.environ.get('XDG_DATA_HOME', home / '.local/share'))]
data_dirs += [Path(p) for p in os.environ.get('XDG_DATA_DIRS', '/usr/local/share:/usr/share').split(':') if p]
manifest_dirs = {p / 'vulkan' / 'icd.d' for p in config_dirs + data_dirs}
manifest_dirs.update((Path('/etc/vulkan/icd.d'), Path('/usr/local/etc/vulkan/icd.d')))
manifest_paths = set()
for var in ('VK_DRIVER_FILES', 'VK_ICD_FILENAMES', 'VK_ADD_DRIVER_FILES'):
    for entry in os.environ.get(var, '').split(':'):
        if not entry:
            continue
        path = Path(entry)
        if path.is_dir():
            manifest_paths.update(path.glob('*.json'))
        elif path.is_file() and path.suffix == '.json':
            manifest_paths.add(path)
for directory in manifest_dirs:
    if directory.is_dir():
        manifest_paths.update(directory.glob('*.json'))


def resolve_library(library_path, manifest_path=None):
    lib = Path(library_path)
    if lib.is_absolute():
        return lib.resolve() if lib.is_file() else None
    if '/' in library_path and manifest_path is not None:
        relative = (manifest_path.parent / lib).resolve()
        if relative.is_file():
            return relative
    for candidate in library_paths.get(library_path, []):
        if candidate.is_file():
            return candidate
    return None

nvidia_manifests = []
invalid_nvidia_manifests = []
for path in sorted(manifest_paths):
    try:
        document = json.loads(path.read_text(encoding='utf-8'))
        icd = document.get('ICD', {})
        library = str(icd.get('library_path', ''))
        is_nvidia = 'nvidia' in (path.name + ' ' + library).lower()
        resolved = resolve_library(library, path) if library else None
        print('ICD manifest:', path, '->', library or '<no ICD library_path>', 'resolved:', resolved or '<missing>')
        if is_nvidia:
            if resolved:
                nvidia_manifests.append(path.resolve())
            else:
                invalid_nvidia_manifests.append(path.resolve())
    except Exception as exc:
        print('Could not read ICD manifest:', path, repr(exc))

# Existing valid NVIDIA manifests and loader settings are preserved. If the
# GPU driver is exactly 580.82.07 but its matching Vulkan userspace is absent,
# optionally unpack NVIDIA's checksum-pinned runfile into this notebook's local
# work directory. --extract-only never runs the driver installer.
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def add_driver_manifest(path):
    additions = [p for p in os.environ.get('VK_ADD_DRIVER_FILES', '').split(':') if p]
    if str(path) not in additions:
        additions.append(str(path))
    os.environ['VK_ADD_DRIVER_FILES'] = ':'.join(additions)


def recover_matching_nvidia_userspace():
    driver_versions = set()
    for row in smi_output.splitlines():
        fields = [field.strip() for field in row.split(',')]
        if len(fields) >= 3:
            driver_versions.add(fields[2])
    if driver_versions != {NVIDIA_DRIVER_VERSION}:
        return None, (
            f'Private recovery is pinned only to driver {NVIDIA_DRIVER_VERSION}; '
            f'nvidia-smi reported {sorted(driver_versions)}. No download or driver change was made.')

    runfile = RECOVERY_WORK / 'NVIDIA-Linux-x86_64-580.82.07-no-compat32.run'
    partial = runfile.with_suffix('.download')
    if not runfile.is_file():
        import urllib.request
        hasher = hashlib.sha256()
        try:
            request = urllib.request.Request(NVIDIA_RUNFILE_URL, headers={
                'User-Agent': 'Spirula-Colab-Vulkan-recovery',
            })
            with urllib.request.urlopen(request, timeout=60) as response, partial.open('wb') as output:
                while True:
                    block = response.read(1024 * 1024)
                    if not block:
                        break
                    output.write(block)
                    hasher.update(block)
            if partial.stat().st_size != NVIDIA_RUNFILE_SIZE:
                raise RuntimeError(f'Runfile size mismatch: {partial.stat().st_size} bytes.')
            if hasher.hexdigest() != NVIDIA_RUNFILE_SHA256:
                raise RuntimeError(f'Runfile SHA-256 mismatch: {hasher.hexdigest()}.')
            partial.replace(runfile)
        except Exception:
            partial.unlink(missing_ok=True)
            raise
    else:
        if runfile.stat().st_size != NVIDIA_RUNFILE_SIZE or sha256_file(runfile) != NVIDIA_RUNFILE_SHA256:
            raise RuntimeError('Cached NVIDIA runfile failed pinned size/SHA-256 verification.')

    extract_parent = RECOVERY_WORK / 'nvidia_extract'
    extract_parent.mkdir(parents=True, exist_ok=True)
    extracted = extract_parent / 'NVIDIA-Linux-x86_64-580.82.07-no-compat32'
    official_manifest = extracted / 'nvidia_icd.json'
    if not official_manifest.is_file():
        code, output = run_diagnostic(['sh', str(runfile), '--extract-only'],
                                      timeout=600, cwd=extract_parent)
        if code:
            raise RuntimeError(f'NVIDIA --extract-only failed ({code}); see {RECOVERY_LOG}.')
    if not official_manifest.is_file():
        raise RuntimeError(f'Pinned archive did not produce its official ICD manifest: {official_manifest}')

    document = json.loads(official_manifest.read_text(encoding='utf-8'))
    icd = document.get('ICD', {})
    if (document.get('file_format_version') != '1.0.1' or
            icd.get('library_path') != 'libGLX_nvidia.so.0' or
            icd.get('api_version') != '1.4.312'):
        raise RuntimeError('Extracted NVIDIA ICD manifest did not match the pinned 580.82.07 package.')

    private_libs = RECOVERY_WORK / 'private_graphics'
    private_libs.mkdir(parents=True, exist_ok=True)
    whitelist = [
        'libEGL_nvidia.so.580.82.07',
        'libnvidia-glsi.so.580.82.07',
        'libnvidia-eglcore.so.580.82.07',
        'libnvidia-glcore.so.580.82.07',
        'libnvidia-glvkspirv.so.580.82.07',
        'libnvidia-gpucomp.so.580.82.07',
        'libnvidia-allocator.so.580.82.07',
        'libnvidia-tls.so.580.82.07',
        'libnvidia-cfg.so.580.82.07',
        'libnvidia-rtcore.so.580.82.07',
    ]
    missing = [name for name in whitelist if not (extracted / name).is_file()]
    if missing:
        raise RuntimeError('Pinned archive lacks required NVIDIA graphics libraries: ' + ', '.join(missing))
    for name in whitelist:
        shutil.copy2(extracted / name, private_libs / name)

    egl = private_libs / 'libEGL_nvidia.so.580.82.07'
    existing_library_paths = [p for p in os.environ.get('LD_LIBRARY_PATH', '').split(':') if p]
    private_library_path = ':'.join([str(private_libs)] + [p for p in existing_library_paths if p != str(private_libs)])
    check_env = os.environ.copy()
    # Prepend extracted graphics userspace while retaining runtime-provided paths.
    # The private whitelist excludes libcuda and NVML.
    check_env['LD_LIBRARY_PATH'] = private_library_path
    ldd = shutil.which('ldd')
    if not ldd:
        raise RuntimeError('ldd is unavailable; cannot check private NVIDIA graphics dependencies safely.')
    ldd_code, ldd_output = run_diagnostic([ldd, str(egl)], show=False, timeout=30, env=check_env)
    if ldd_code or 'not found' in ldd_output.lower():
        raise RuntimeError(f'Private NVIDIA EGL dependencies did not resolve; see {RECOVERY_LOG}.')

    # Preserve NVIDIA's verified API version and format; point its official ICD at EGL,
    # which works in headless runtimes without depending on the X11 GLX path.
    document['ICD']['library_path'] = str(egl.resolve())
    local_manifest = RECOVERY_WORK / 'vulkan' / 'nvidia_icd.json'
    local_manifest.parent.mkdir(parents=True, exist_ok=True)
    local_manifest.write_text(json.dumps(document, indent=2), encoding='utf-8')
    add_driver_manifest(local_manifest)
    os.environ['LD_LIBRARY_PATH'] = private_library_path
    print('Installed matching NVIDIA Vulkan userspace privately under:', private_libs)
    print('Only --extract-only ran; no kernel driver, system driver package, libcuda, or NVML was installed.')
    return local_manifest, None


local_manifest = None
private_recovery = False
if not nvidia_manifests:
    icd_library = library_paths['libEGL_nvidia.so.0'] or library_paths['libGLX_nvidia.so.0']
    if not icd_library:
        try:
            local_manifest, blocker = recover_matching_nvidia_userspace()
        except Exception as exc:
            raise RuntimeError(f'Pinned private NVIDIA Vulkan recovery failed: {exc}. Full diagnostics: {RECOVERY_LOG}') from exc
        if local_manifest:
            private_recovery = True
            nvidia_manifests.append(local_manifest.resolve())
        else:
            raise RuntimeError(
                'nvidia-smi sees a GPU, but no NVIDIA Vulkan ICD library or valid NVIDIA ICD manifest exists. '
                + str(blocker) + ' Do not install or upgrade the GPU driver here; select a runtime exposing matching Vulkan userspace. '
                + f'Full diagnostics: {RECOVERY_LOG}')

if nvidia_manifests:
    if private_recovery:
        print('Using checksum-verified, process-local NVIDIA Vulkan userspace and ICD manifest:', local_manifest)
    else:
        print('Existing valid NVIDIA ICD manifest found; leaving Vulkan loader configuration intact.')
elif invalid_nvidia_manifests:
    raise RuntimeError('An NVIDIA ICD manifest exists but its library path is missing or invalid. No override was made. Inspect the local manifest paths printed above and use a runtime with matching NVIDIA Vulkan userspace libraries.')
else:
    icd_library = library_paths['libEGL_nvidia.so.0'] or library_paths['libGLX_nvidia.so.0']
    local_manifest = RECOVERY_WORK / 'vulkan' / 'nvidia_icd.json'
    local_manifest.parent.mkdir(parents=True, exist_ok=True)
    manifest_document = {
        'file_format_version': '1.0.1',
        'ICD': {'library_path': str(icd_library[0]), 'api_version': '1.0.0', 'library_arch': '64'},
    }
    local_manifest.write_text(json.dumps(manifest_document, indent=2), encoding='utf-8')
    add_driver_manifest(local_manifest)
    print('Created process-local NVIDIA ICD manifest:', local_manifest)
    print('Added it with VK_ADD_DRIVER_FILES; this is inherited by the trainer process.')

vulkaninfo = shutil.which('vulkaninfo')
if not vulkaninfo:
    raise RuntimeError(f'vulkaninfo is missing. Install only vulkan-tools if needed; see {RECOVERY_LOG}.')
info_code, vulkan_summary = run_diagnostic([vulkaninfo, '--summary'])

def nvidia_vulkan_devices(summary):
    blocks, current = [], []
    for line in summary.splitlines():
        if re.match(r'^\s*GPU\s*\d+:\s*$', line):
            if current:
                blocks.append('\n'.join(current))
            current = [line]
        elif current:
            current.append(line)
    if current:
        blocks.append('\n'.join(current))
    devices = []
    for block in blocks:
        def field(name):
            match = re.search(r'^\s*' + re.escape(name) + r'\s*=\s*(.*?)\s*$', block, re.MULTILINE)
            return match.group(1).strip() if match else ''
        vendor = field('vendorID').lower()
        kind = field('deviceType')
        name = field('deviceName')
        device_uuid = re.sub(r'[^0-9a-fA-F]', '', field('deviceUUID')).lower()
        if vendor in ('0x10de', '4318') and 'DISCRETE_GPU' in kind and len(device_uuid) == 32:
            devices.append({'name': name, 'vendor_id': vendor, 'type': kind, 'uuid': device_uuid})
    return devices

found = nvidia_vulkan_devices(vulkan_summary)
if info_code or not found:
    raise RuntimeError(
        f'Vulkan did not enumerate a usable NVIDIA discrete GPU (vulkaninfo exit={info_code}, '
        f'valid NVIDIA GPUs={len(found)}). No CPU/software fallback is permitted. '
        f'Inspect {RECOVERY_LOG}; the running kernel keeps this diagnostic in {RECOVERY_WORK}.')
GPU_NAME = found[0]['name']
GPU_SELECTOR = 'uuid:' + found[0]['uuid']
if isinstance(globals().get('manifest'), dict):
    manifest['vulkan_gpu'] = found[0]
    manifest['vulkan_icd_manifest'] = str(local_manifest) if 'local_manifest' in globals() else 'existing'
print('Verified NVIDIA Vulkan GPU:', GPU_NAME, GPU_SELECTOR)
print('Use GPU_SELECTOR as --device for Spirula; the current kernel inherits any additive ICD setting.')


## 4. Kaynağı yerel diske al, kareleri çıkar ve SfM bileşenlerini denetle


In [ ]:
# Check GPU memory before expensive preprocessing.
nvidia_smi = shutil.which('nvidia-smi')
require(nvidia_smi is not None, 'nvidia-smi GPU görünmüyor.', 'train_preflight')
memory_text = run_logged('train_memory_preflight', [nvidia_smi, '--query-gpu=memory.total',
                          '--format=csv,noheader,nounits'], timeout=30)
memory_lines = [line.strip() for line in memory_text.splitlines()
                if line.strip().replace('MiB', '').strip().isdigit()]
require(memory_lines, f'nvidia-smi sayısal bellek döndürmedi: {memory_text[-500:]}', 'train_preflight')
total_mib = int(memory_lines[-1].replace('MiB', '').strip())
gpu_gib = total_mib / 1024.0
manifest['train_preflight'] = {'gpu_memory_mib': total_mib, 'gpu_memory_gib': round(gpu_gib, 2),
                               'required_gib': MIN_TRAIN_VRAM_GIB, 'cap_max': TRAIN_CAP_MAX,
                               'iterations': TRAIN_ITERATIONS}
if gpu_gib < MIN_TRAIN_VRAM_GIB:
    manifest['status'] = 'train_preflight_blocked'
    manifest['error'] = (f'Seçilen eğitim için {MIN_TRAIN_VRAM_GIB} GiB gerekir; '
                         f'GPU {gpu_gib:.2f} GiB bildiriyor. Bütçe otomatik azaltılmadı.')
    save_manifest()
    fail(manifest['error'] + ' Daha küçük GPU için TRAIN_CAP_MAX değerini ve MIN_TRAIN_VRAM_GIB eşiğini ayar hücresinde birlikte düşür.',
         'train_preflight')
print(f'GPU bellek preflight: {gpu_gib:.2f} GiB; eğitim bütçesi {TRAIN_CAP_MAX:,} splat / {TRAIN_ITERATIONS:,} adım.')

import math
import re
import struct
import zipfile
from fractions import Fraction
from pathlib import PurePosixPath



def sync_dataset_snapshot():
    dataset_root = globals().get('DATASET')
    if not isinstance(dataset_root, Path) or not dataset_root.is_dir():
        return []
    geometry_root = globals().get('GEOMETRY_DATASET') or dataset_root
    target = DRIVE_OUT / 'diagnostic_dataset'
    sources = [('images', dataset_root / 'images'),
               ('sparse', geometry_root / 'sparse'),
               ('sparse_components', dataset_root / 'sparse'),
               ('normals', geometry_root / 'normals'),
               ('depths', geometry_root / 'depths')]
    errors = []
    for name, source in sources:
        if source.is_dir():
            ok, error = _copy_tree_to_drive(source, target / name, 'diagnostic_dataset/' + name)
            if not ok:
                errors.append(error)
    report = dataset_root / 'sfm_report.json'
    if report.is_file():
        ok, error = _copy_file_to_drive(report, target / 'sfm_report.json', 'diagnostic_dataset/sfm_report.json')
        if not ok:
            errors.append(error)
    manifest['diagnostic_dataset_local'] = str(dataset_root)
    manifest['diagnostic_dataset_drive'] = str(target)
    manifest['diagnostic_dataset_sync_errors'] = errors
    save_manifest()
    return errors

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.tif', '.tiff'}
DATASET = WORK / 'dataset'
RECON_DIR = None
IMAGE_DIR = None
SFM_REUSED = False


def check_free_space(required_bytes, label):
    free_bytes = shutil.disk_usage('/content').free
    require(required_bytes + 2 * 1024**3 < free_bytes,
            f'{label} için yeterli /content alanı yok. Gerekli yaklaşık bayt: {required_bytes + 2 * 1024**3}; boş: {free_bytes}.',
            'input')


def copy_input_folder(source, destination):
    files = [p for p in source.rglob('*') if p.is_file()]
    total = sum(p.stat().st_size for p in files)
    check_free_space(total, 'Dataset kopyası')
    shutil.copytree(source, destination, dirs_exist_ok=True)


def safe_extract_dataset(source_zip, destination):
    with zipfile.ZipFile(source_zip) as zf:
        entries = zf.infolist()
        total = sum(item.file_size for item in entries)
        check_free_space(total, 'Dataset ZIP açma')
        root = destination.resolve()
        for item in entries:
            member = PurePosixPath(item.filename.replace('\\', '/'))
            target = (destination / Path(*member.parts)).resolve()
            require(not member.is_absolute() and '..' not in member.parts and target.is_relative_to(root),
                    f'ZIP içinde geçersiz yol: {item.filename}', 'input')
            require(not stat.S_ISLNK(item.external_attr >> 16),
                    f'ZIP sembolik bağlantı içeriyor: {item.filename}', 'input')
        zf.extractall(destination)


def discover_dataset_root(base):
    candidates = sorted({p.parent for p in base.rglob('images') if p.is_dir() and
                         any(f.is_file() and f.suffix.lower() in IMAGE_SUFFIXES for f in p.rglob('*'))})
    require(len(candidates) == 1,
            f'Tek bir images/ klasörü içeren dataset bekleniyordu; bulunanlar: {candidates}', 'input')
    return candidates[0]


if INPUT_MODE == 'video':
    local_video = WORK / ('source_video' + VIDEO_PATH.suffix.lower())
    try:
        size = VIDEO_PATH.stat().st_size
        check_free_space(size, 'Video kopyası')
        shutil.copy2(VIDEO_PATH, local_video)
    except Exception as exc:
        fail(f'Video Drive’dan yerel diske alınamadı: {type(exc).__name__}: {exc}', 'input')
    DATASET.mkdir(parents=True, exist_ok=False)
    IMAGE_DIR = DATASET / 'images'
    IMAGE_DIR.mkdir()
    run_logged('video_info', [str(BINARY), 'sam', 'video', '--info', str(local_video)], cwd=WORK, timeout=120)
    if FRAME_SKIP_OVERRIDE is not None:
        frame_skip = int(FRAME_SKIP_OVERRIDE)
        require(frame_skip >= 1, 'FRAME_SKIP_OVERRIDE en az 1 olmalı.', 'input')
        source_fps = None
    else:
        ffprobe = shutil.which('ffprobe')
        require(ffprobe is not None,
                'Video örneklemesi için ffprobe gerekli. ffmpeg/ffprobe kur veya FRAME_SKIP_OVERRIDE değerini ayarla.',
                'input')
        probe_text = run_logged('video_frame_rate', [ffprobe, '-v', 'error', '-select_streams', 'v:0',
                                '-show_entries', 'stream=avg_frame_rate,r_frame_rate', '-of', 'json',
                                str(local_video)], cwd=WORK, timeout=120)
        try:
            # run_logged prefixes the command on line one; ffprobe JSON follows.
            probe_payload = probe_text.split('\n', 1)[1]
            streams = json.loads(probe_payload)['streams']
            raw_fps = streams[0].get('avg_frame_rate') or streams[0].get('r_frame_rate')
            source_fps = float(Fraction(raw_fps))
            require(math.isfinite(source_fps) and source_fps > 0, f'Video fps değeri geçersiz: {raw_fps}', 'input')
            frame_skip = max(1, round(source_fps / float(TARGET_FPS)))
        except Exception as exc:
            fail(f'Video kare hızı çözümlenemedi: {exc}. FRAME_SKIP_OVERRIDE değerini ayarla.', 'input')
    extract_args = [str(BINARY), 'sam', 'extract', str(local_video), '--out', str(IMAGE_DIR),
                    '--skip', str(frame_skip), '--quality', '101', '--adaptive', '--device', GPU_SELECTOR,
                    '--lang', 'en']
    if MAX_EXTRACTED_FRAMES > 0:
        extract_args += ['--max-frames', str(int(MAX_EXTRACTED_FRAMES))]
    manifest['video_sampling'] = {'source_fps': source_fps, 'target_fps': TARGET_FPS,
                                  'frame_skip': frame_skip, 'adaptive': True,
                                  'format': 'PNG', 'frame_limit': MAX_EXTRACTED_FRAMES}
    run_logged('frame_extract', extract_args, cwd=WORK, timeout=7200)
elif INPUT_MODE == 'dataset_zip':
    staging = WORK / 'input_zip'
    staging.mkdir()
    safe_extract_dataset(DATASET_ZIP, staging)
    source_root = discover_dataset_root(staging)
    copy_input_folder(source_root, DATASET)
else:
    require(DATASET_FOLDER.is_dir(), f'Dataset klasörü bulunamadı: {DATASET_FOLDER}', 'input')
    copy_input_folder(DATASET_FOLDER, DATASET)

if INPUT_MODE != 'video':
    IMAGE_DIR = DATASET / 'images'
require(IMAGE_DIR.is_dir(), f'Görüntü klasörü bulunamadı: {IMAGE_DIR}', 'input')
image_files = sorted(p for p in IMAGE_DIR.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
require(len(image_files) >= 10, f'En az 10 görüntü bekleniyordu; bulundu: {len(image_files)}.', 'input')
manifest.update({'local_dataset': str(DATASET), 'image_dir': str(IMAGE_DIR), 'input_image_count': len(image_files)})
save_manifest()
print('Yerel dataset:', DATASET)
print('SfM girişi:', len(image_files), 'görüntü')

# A prepared dataset may already carry COLMAP cameras. Video input always runs
# extreme SfM; otherwise reuse valid cameras or reconstruct from images.
recon_root = DATASET / 'sparse'
existing_models = sorted(p for p in recon_root.iterdir() if p.is_dir() and (p / 'images.bin').is_file()) if recon_root.is_dir() else []
if INPUT_MODE == 'video' or not existing_models:
    capture_kind = 'video' if INPUT_MODE == 'video' else CAPTURE_KIND
    require(capture_kind in ('video', 'individual'), 'CAPTURE_KIND video veya individual olmalı.', 'sfm')
    sfm_args = [str(BINARY), 'sfm', 'auto', str(IMAGE_DIR), '--output', str(DATASET),
                '--quality', SFM_QUALITY, '--data-type', capture_kind, '--pairs', 'auto',
                '--device', GPU_SELECTOR, '--lang', 'en']
    if capture_kind == 'video':
        sfm_args += ['--sequence', '.', '--prefilter-sequential']
    manifest['sfm_command'] = sfm_args
    save_manifest()
    try:
        sfm_log = run_logged('sfm_extreme', sfm_args, cwd=WORK, timeout=21600, accepted_exit_codes=(0, 3))
        require('RESULT:' in sfm_log and any((DATASET / 'sparse').glob('*/images.bin')),
                'SfM usable result/model could not be verified; inspect sfm_extreme.log.', 'sfm')
        if 'RESULT: PARTIAL' in sfm_log:
            print('SfM returned a partial reconstruction; validate the written models and apply the quality gate.')
            manifest['sfm_partial'] = True
            save_manifest()
    except Exception:
        raise
else:
    SFM_REUSED = True
    print('COLMAP sparse model bulundu; mevcut model kullanılıyor. SfM sıfırdan istenirse video girişiyle çalıştır.')


def registered_image_names(images_bin):
    names = []
    with images_bin.open('rb') as stream:
        raw = stream.read(8)
        require(len(raw) == 8, f'COLMAP başlığı bozuk: {images_bin}', 'sfm')
        count = struct.unpack('<Q', raw)[0]
        require(0 < count <= 1000000, f'COLMAP image sayısı mantıksız: {count}', 'sfm')
        for _ in range(count):
            require(len(stream.read(64)) == 64, f'COLMAP kaydı kesik: {images_bin}', 'sfm')
            name_bytes = bytearray()
            while len(name_bytes) < 4096:
                char = stream.read(1)
                require(char, f'COLMAP görüntü adı bozuk: {images_bin}', 'sfm')
                if char == b'\x00':
                    break
                name_bytes.extend(char)
            else:
                fail(f'COLMAP görüntü adı 4096 baytı aşıyor: {images_bin}', 'sfm')
            names.append(name_bytes.decode('utf-8'))
            point_bytes = stream.read(8)
            require(len(point_bytes) == 8, f'COLMAP 2D nokta başlığı eksik: {images_bin}', 'sfm')
            point_count = struct.unpack('<Q', point_bytes)[0]
            require(stream.tell() + point_count * 24 <= images_bin.stat().st_size,
                    f'COLMAP 2D nokta kaydı bozuk: {images_bin}', 'sfm')
            stream.seek(point_count * 24, 1)
    return names


recon_dirs = sorted(p for p in (DATASET / 'sparse').glob('*') if p.is_dir() and (p / 'images.bin').is_file())
require(recon_dirs, f'SfM sparse model oluşturmadı: {DATASET / "sparse"}', 'sfm')
component_names = {str(path): registered_image_names(path / 'images.bin') for path in recon_dirs}
registered_union = set().union(*(set(names) for names in component_names.values()))
image_name_set = {p.relative_to(IMAGE_DIR).as_posix() for p in image_files}
total_registered_count = len(registered_union & image_name_set)
component_counts = {str(path): len(set(names) & image_name_set) for path, names in
                    ((Path(k), v) for k, v in component_names.items())}
ranked_components = sorted(component_counts.items(), key=lambda kv: kv[1], reverse=True)
RECON_DIR = Path(ranked_components[0][0])
registered_count = ranked_components[0][1]
coverage = registered_count / len(image_files)
dominant_share = registered_count / max(total_registered_count, 1)
sfm_report = {'input_images': len(image_files), 'registered_images': registered_count,
              'total_registered_across_components': total_registered_count, 'coverage': coverage,
              'component_count': len(ranked_components),
              'component_registered_images': ranked_components, 'dominant_component_share': dominant_share,
              'selected_reconstruction': str(RECON_DIR)}
(DATASET / 'sfm_report.json').write_text(json.dumps(sfm_report, indent=2), encoding='utf-8')
manifest['sfm'] = sfm_report
save_manifest()
print(f'SfM: seçilen en büyük model {registered_count}/{len(image_files)} görüntü ({coverage:.1%}); '
      f'{len(ranked_components)} bileşen; bu model toplam kayıtlı görüntülerin {dominant_share:.1%} kadarını kapsıyor.')
print('Seçilen COLMAP modeli:', RECON_DIR)

# GEOMETRY_DATASET is created by the separate, rerunnable quality-gate cell.
GEOMETRY_DATASET = None
TRAIN_DATASET = None
TRAIN_RECON_DIR = None


## 4a. SfM kalite kapısı ve seçilen modeli bağla


Bu hücre preprocessing tamamlandıktan sonra ayrı çalışır. Kapı durdurursa kareler, SfM raporu ve bileşenler korunur. Açıkça devam etmek için yeni bir kod hücresinde `ALLOW_PARTIAL_RECONSTRUCTION = True` çalıştırıp yalnız bu kapı hücresini yeniden çalıştır; preprocessing hücresini baştan çalıştırma.


In [ ]:
if (coverage < MIN_REGISTERED_COVERAGE or dominant_share < MIN_DOMINANT_COMPONENT_SHARE) and not ALLOW_PARTIAL_RECONSTRUCTION:
    manifest['status'] = 'reconstruction_incomplete'
    manifest['error'] = ('SfM kapsamı yetersiz: düşük kayıt oranı veya parçalı model. '
                         'Ham kareleri ve sparse bileşenlerini Drive diagnostics klasörüne aktarıyorum.')
    save_manifest()
    fail(f'Uzun eğitim durduruldu: kayıt kapsamı {coverage:.1%} (eşik {MIN_REGISTERED_COVERAGE:.0%}), '
         f'en büyük bileşen payı {dominant_share:.1%} (eşik {MIN_DOMINANT_COMPONENT_SHARE:.0%}). '
         'Raporu ve her bileşeni incele; devam etmek istiyorsan yeni kod hücresinde '
         'ALLOW_PARTIAL_RECONSTRUCTION = True çalıştırıp yalnız bu kapı hücresini yeniden çalıştır.',
         'sfm_quality_gate')

# Geometry and training use the same chosen largest model as sparse/0. Idempotent
# so this gate cell can be rerun without re-extracting frames or rerunning SfM.
GEOMETRY_DATASET = WORK / 'geometry_dataset'
GEOMETRY_DATASET.mkdir(exist_ok=True)
sparse_root = GEOMETRY_DATASET / 'sparse'
sparse_root.mkdir(exist_ok=True)

def ensure_directory_link(link, target):
    target = Path(target).resolve()
    if link.is_symlink():
        require(link.resolve() == target, f'Eski dataset bağlantısı beklenen hedefle eşleşmiyor: {link}', 'sfm_quality_gate')
    elif link.exists():
        require(link.resolve() == target, f'Dataset bağlantı yolunda beklenmeyen dosya/klasör var: {link}', 'sfm_quality_gate')
    else:
        link.symlink_to(target, target_is_directory=True)

ensure_directory_link(GEOMETRY_DATASET / 'images', IMAGE_DIR)
ensure_directory_link(sparse_root / '0', RECON_DIR)
TRAIN_DATASET = GEOMETRY_DATASET
TRAIN_RECON_DIR = Path('sparse') / '0'
manifest['status'] = 'sfm_complete'
manifest.pop('error', None)
save_manifest()
print(f'SfM kalite kapısı geçti: kapsam {coverage:.1%}, en büyük bileşen payı {dominant_share:.1%}.')
print('Eğitim veri görünümü:', TRAIN_DATASET, '| COLMAP modeli:', TRAIN_RECON_DIR)


## 5. Geometri modeli: yüzey normalleri ve göreli derinlik


In [ ]:
geometry_args = [str(BINARY), 'geometry', str(GEOMETRY_DATASET), '--model', GEOMETRY_MODEL,
                 '--max-size', str(GEOMETRY_MAX_SIZE), '--num-tokens', str(GEOMETRY_NUM_TOKENS),
                 '--normal-format', 'png', '--overwrite', '--device', GPU_SELECTOR, '--lang', 'en']
if GENERATE_DEPTH:
    geometry_args += ['--depth', '--depth-units', 'relative']
manifest['geometry_command'] = geometry_args
manifest['geometry'] = {'model': GEOMETRY_MODEL, 'max_size': GEOMETRY_MAX_SIZE,
                        'num_tokens': GEOMETRY_NUM_TOKENS, 'depth': GENERATE_DEPTH,
                        'normal_format': 'png'}
save_manifest()
try:
    run_logged('geometry_max', geometry_args, cwd=WORK, timeout=21600)
except RuntimeError as exc:
    if "process exited with code -11" not in str(exc):
        raise
    # Run in a NEW cell in the same Colab session. Does not rerun geometry.
    from pathlib import Path
    import struct, re
    import cv2
    import numpy as np
    assert 'GEOMETRY_DATASET' in globals() and GEOMETRY_DATASET is not None, 'Run cell 4a first.'
    assert 'manifest' in globals() and 'LOGS' in globals(), 'Use the existing notebook session.'

    def registered_image_names(images_bin):
        names = []
        with images_bin.open('rb') as stream:
            raw = stream.read(8)
            require(len(raw) == 8, f'COLMAP başlığı bozuk: {images_bin}', 'sfm')
            count = struct.unpack('<Q', raw)[0]
            require(0 < count <= 1000000, f'COLMAP image sayısı mantıksız: {count}', 'sfm')
            for _ in range(count):
                require(len(stream.read(64)) == 64, f'COLMAP kaydı kesik: {images_bin}', 'sfm')
                name_bytes = bytearray()
                while len(name_bytes) < 4096:
                    char = stream.read(1)
                    require(char, f'COLMAP görüntü adı bozuk: {images_bin}', 'sfm')
                    if char == b'\x00':
                        break
                    name_bytes.extend(char)
                else:
                    fail(f'COLMAP görüntü adı 4096 baytı aşıyor: {images_bin}', 'sfm')
                names.append(name_bytes.decode('utf-8'))
                point_bytes = stream.read(8)
                require(len(point_bytes) == 8, f'COLMAP 2D nokta başlığı eksik: {images_bin}', 'sfm')
                point_count = struct.unpack('<Q', point_bytes)[0]
                require(stream.tell() + point_count * 24 <= images_bin.stat().st_size,
                        f'COLMAP 2D nokta kaydı bozuk: {images_bin}', 'sfm')
                stream.seek(point_count * 24, 1)
        return names


    def validate_geometry_maps(root, names, require_depth):
        problems = []
        counts = {'normals': 0, 'depths': 0}
        for name in names:
            relative = Path(name.replace(chr(92), '/'))
            if relative.is_absolute() or '..' in relative.parts:
                raise RuntimeError(f'Unsafe image name: {name}')
            normal_shape = None
            for kind in ('normals', 'depths') if require_depth else ('normals',):
                # This run requested PNG normals; depth maps are PNG too.
                path = root / kind / relative.with_suffix('.png')
                image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED) if path.is_file() else None
                if image is None or image.size == 0:
                    problems.append(f'Missing/unreadable: {path}')
                    continue
                if not np.isfinite(image).all():
                    problems.append(f'Nonfinite map: {path}')
                    continue
                if kind == 'normals':
                    if image.ndim != 3 or image.shape[2] not in (3, 4):
                        problems.append(f'Invalid normal shape: {path}: {image.shape}')
                        continue
                    normal_shape = image.shape[:2]
                else:
                    if image.ndim != 2 or image.dtype != np.uint16 or not np.any(image > 0):
                        problems.append(f'Invalid/empty 16-bit depth map: {path}')
                        continue
                    if normal_shape is not None and image.shape != normal_shape:
                        problems.append(f'Normal/depth size mismatch: {path}')
                        continue
                counts[kind] += 1
        if problems:
            raise RuntimeError('Geometry validation failed; do not start training.\n' + '\n'.join(problems[:20]))
        return counts

    root = Path(GEOMETRY_DATASET)
    log_text = (LOGS / 'geometry_max.log').read_text(encoding='utf-8', errors='replace')
    assert re.search(r'done:\s*\d+ written', log_text), 'Geometry completion marker not found.'
    names = registered_image_names(root / 'sparse' / '0' / 'images.bin')
    assert names and len(names) == len(set(names)), 'Invalid/duplicate COLMAP image names.'
    counts = validate_geometry_maps(root, names, GENERATE_DEPTH)
    manifest.setdefault('geometry', {}).update({
        'normal_map_count': counts['normals'], 'depth_map_count': counts['depths'],
        'verified_registered_images': len(names), 'file_validation_passed': True,
        'recovered_after_process_crash': True})
    manifest['geometry_warning'] = 'Original geometry process crashed. All selected image maps were decoded and validated; crash cause remains unresolved.'
    manifest['status'] = 'geometry_outputs_validated_after_crash'
    for key in ('error', 'failed_step', 'local_failed_log'):
        manifest.pop(key, None)
    save_manifest()
    sync_dataset_snapshot()
    print(f"VALIDATED: {counts['normals']} normal maps, {counts['depths']} depth maps for {len(names)} images.")


normal_files = sorted(p for p in (GEOMETRY_DATASET / 'normals').rglob('*') if p.is_file()) if (GEOMETRY_DATASET / 'normals').is_dir() else []
depth_files = sorted(p for p in (GEOMETRY_DATASET / 'depths').rglob('*') if p.is_file()) if (GEOMETRY_DATASET / 'depths').is_dir() else []
manifest['geometry'].update({'normal_map_count': len(normal_files), 'depth_map_count': len(depth_files)})
require(len(normal_files) >= max(1, int(0.8 * ranked_components[0][1])),
        f'Geometri normal haritaları eksik: {len(normal_files)} dosya / {registered_count} kayıtlı görüntü.', 'geometry')
if GENERATE_DEPTH:
    require(len(depth_files) >= max(1, int(0.8 * ranked_components[0][1])),
            f'Geometri depth haritaları eksik: {len(depth_files)} dosya / {registered_count} kayıtlı görüntü.', 'geometry')
save_manifest()
sync_dataset_snapshot()
print('Normal haritaları:', len(normal_files), '| derinlik haritaları:', len(depth_files))


## 6. Bellek eşiğini doğrula ve seçilen eğitimi çalıştır


In [ ]:
RUN_OUTPUT = OUT / 'training'
resume_step = 0
if RESUME_CHECKPOINT:
    resume_path = Path(RESUME_CHECKPOINT)
    require(resume_path.is_dir() and re.fullmatch(r'step-(\d{9})\.ckpt', resume_path.name),
            'RESUME_CHECKPOINT tam bir step-XXXXXXXXX.ckpt klasörünü göstermeli.', 'resume')
    resume_step = int(re.search(r'step-(\d{9})', resume_path.name).group(1))
    require(0 < resume_step < TRAIN_ITERATIONS,
            f'Resume checkpoint adımı {resume_step}; hedef {TRAIN_ITERATIONS}.', 'resume')

args = [str(BINARY), 'train', '3dgs', '--data', str(TRAIN_DATASET), '--colmap-recon-dir', str(TRAIN_RECON_DIR),
        '--output-dir-prefix', str(OUT), '--output-dir-name', 'training',
        '--quality', TRAIN_QUALITY, '--num-iterations', str(TRAIN_ITERATIONS),
        '--max-steps', str(TRAIN_ITERATIONS), '--cap-max', str(TRAIN_CAP_MAX),
        '--steps-per-save', str(CHECKPOINT_INTERVAL), '--save-full-checkpoint', 'true',
        '--save-only-latest-checkpoint', 'true', '--train-resolution-divisor', '1', '--sh-degree', '3',
        '--cache-images', 'disk', '--disable-viewer', 'true', '--keep-viewer-alive', 'false',
        '--load-normals', 'true', '--normal-supervision-weight', '0.01',
        '--load-depths', str(GENERATE_DEPTH).lower(), '--depth-supervision-weight', '0.01' if GENERATE_DEPTH else '0',
        '--device', GPU_SELECTOR, '--lang', 'en']
if RESUME_CHECKPOINT:
    args += ['--resume', str(Path(RESUME_CHECKPOINT))]


def mirror_completed_checkpoint(step):
    checkpoint = RUN_OUTPUT / f'step-{step:09d}.ckpt'
    if not checkpoint.is_dir():
        print('Checkpoint mirror skipped; completed-step path is absent:', checkpoint, flush=True)
        return False
    remote_root = DRIVE_OUT / 'checkpoints'
    destination = remote_root / checkpoint.name
    staging = remote_root / ('.upload-' + checkpoint.name)
    try:
        remote_root.mkdir(parents=True, exist_ok=True)
        if staging.exists():
            shutil.rmtree(staging)
        shutil.copytree(checkpoint, staging)
        if destination.exists():
            shutil.rmtree(destination)
        staging.replace(destination)
        for older in remote_root.glob('step-*.ckpt'):
            if older != destination:
                shutil.rmtree(older)
        manifest['latest_complete_checkpoint'] = {'step': step, 'local': str(checkpoint),
                                                   'drive': str(destination), 'verified_after_progress': True}
        save_manifest()
        print('Drive checkpoint mirror completed after CLI progress confirmed the save:', destination, flush=True)
        return True
    except Exception as exc:
        print(f'Checkpoint Drive mirror failed; local checkpoint remains: {type(exc).__name__}: {exc}', flush=True)
        try:
            if staging.exists():
                shutil.rmtree(staging)
        except Exception:
            pass
        return False


def run_training_with_checkpoint_mirroring(args, total_steps, initial_step):
    log_path = LOGS / 'train_ultra.log'
    proc = None
    problem = None
    last_mirrored = initial_step // CHECKPOINT_INTERVAL * CHECKPOINT_INTERVAL
    with log_path.open('w', encoding='utf-8') as log:
        log.write('COMMAND: ' + repr(args) + '\n')
        log.flush()
        try:
            proc = subprocess.Popen(args, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, env=os.environ.copy())
            for line in proc.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
                match = re.search(r'\b(\d+)\s*/\s*' + str(total_steps) + r'\b', line)
                if not match:
                    continue
                completed = int(match.group(1))
                checkpoint_step = ((completed - 1) // CHECKPOINT_INTERVAL) * CHECKPOINT_INTERVAL
                # The CLI prints step+1 only after that step returns. At counts
                # interval+1, the interval checkpoint saved before that step is complete.
                if checkpoint_step > last_mirrored and checkpoint_step > 0:
                    if mirror_completed_checkpoint(checkpoint_step):
                        last_mirrored = checkpoint_step
            return_code = proc.wait()
            if return_code != 0:
                raise RuntimeError(f'train_ultra exited with code {return_code}')
        except BaseException as exc:
            problem = exc
            if proc is not None and proc.poll() is None:
                proc.terminate()
                try:
                    proc.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait()
            log.write('ERROR: ' + repr(exc) + '\n')
            log.flush()
    log_text = log_path.read_text(encoding='utf-8', errors='replace')
    if problem is not None:
        tail = '\n'.join(log_text.splitlines()[-60:])
        manifest.update({'status': 'failed', 'training_status': 'failed', 'error': str(problem),
                         'local_failed_log': str(log_path)})
        save_manifest()
        retry_drive_export()
        raise RuntimeError(f'{problem}\nLocal log: {log_path}\nLast 60 lines:\n{tail}') from problem
    return log_text

manifest.update({'status': 'training', 'training_status': 'running', 'command': args,
                 'output_folder': str(RUN_OUTPUT), 'resume_step': resume_step,
                 'quality': TRAIN_QUALITY, 'iterations_target': TRAIN_ITERATIONS,
                 'splat_cap': TRAIN_CAP_MAX, 'geometry_model': GEOMETRY_MODEL})
save_manifest()
started = time.monotonic()
train_text = run_training_with_checkpoint_mirroring(args, TRAIN_ITERATIONS, resume_step)
checkpoint = RUN_OUTPUT / f'step-{TRAIN_ITERATIONS:09d}.ckpt'
require(checkpoint.is_dir(), f'Tam hedef checkpoint bulunamadı: {checkpoint}', 'train')
ply_files = sorted(RUN_OUTPUT.rglob('splat.ply'))
require(ply_files, 'Eğitim tamamlandı ancak splat.ply bulunamadı.', 'train')
for ply in ply_files:
    require(ply.stat().st_size > 1024, f'PLY boş veya çok küçük: {ply}', 'train')
    with ply.open('rb') as stream:
        require(stream.read(4) == b'ply\n', f'PLY başlığı bozuk: {ply}', 'train')

manifest.update({'status': 'export_pending', 'training_status': 'complete',
                 'training_seconds': round(time.monotonic() - started, 2),
                 'checkpoint': str(checkpoint.relative_to(OUT)),
                 'ply_files': [{'path': str(p.relative_to(OUT)), 'bytes': p.stat().st_size,
                                'sha256': sha256_file(p)} for p in ply_files]})
save_manifest()
mirror_completed_checkpoint(TRAIN_ITERATIONS)
retry_drive_export()
if manifest.get('status') == 'complete':
    print('80.000 adımlık ultra eğitim ve doğrulanmış Drive aktarımı tamamlandı:', DRIVE_OUT)
    for entry in manifest.get('ply_files', []):
        print('Drive PLY:', DRIVE_OUT / 'outputs' / entry['path'])
else:
    print('Eğitim ve PLY doğrulaması yerelde tamamlandı; Drive aktarımı beklemede.')
    print('Yerel manifest:', MANIFEST_PATH, '| günlükler:', LOGS, '| çıktılar:', OUT)


### Drive’a aktarımı yeniden dene


Drive veya Colab FUSE bağlantısı geri geldiyse bu hücreyi çalıştır. Yerel günlükler, yeniden üretilebilir dataset, çıktılar ve manifest yeniden eşitlenmeye çalışılır; eşitleme hatası yerel dosyaları silmez.


In [ ]:
sync_errors = retry_drive_export()
if sync_errors:
    print('Aktarım hâlâ beklemede. Yerel klasörleri açıp günlükleri inceleyebilirsin:', WORK)
else:
    print('Eşitleme denemesi tamamlandı. Yerel kopya:', WORK)


### Sonraki adım

SfM kapısında durursa `sfm_report.json` ve `sparse/0..N` bileşenlerini incele; daha çok örtüşen kare, daha yavaş kamera hareketi veya daha kısa kamera dönüşüyle yeni video çekimi gerekebilir. Açıkça kısmi modele devam etmeyi seçersen `ALLOW_PARTIAL_RECONSTRUCTION = True` değerini ayrı bir kod hücresinde çalıştırıp yalnız `4a` kalite kapısı hücresini yeniden çalıştır; ön işleme hücresini tekrarlama. PLY yalnızca eğitim 80.000 adıma ulaşıp yerel dosya başlığı ve boyutu doğrulandıktan sonra tamamlandı sayılır. Resume kullanırken aynı görüntüleri, COLMAP modelini, geometri haritalarını ve kalite ayarlarını koru; ayar hücresine Drive’daki tam `step-XXXXXXXXX.ckpt` klasörünü yaz.
